# Multi-Label Classification in Turkish Tweets

Downloads the Turkish tweet dataset from Kaggle and retrieves the path to the dataset files.


In [ ]:
import kagglehub

path = kagglehub.dataset_download("smailsariteke/trke-tweetlerin-6-snfa-gre-etiketlenmesi")

print("Path to dataset files:", path)


Imports the essential base libraries (pandas, numpy) for data processing.


In [ ]:
import pandas as pd
import numpy as np


Reads the downloaded Excel dataset as a pandas DataFrame.


In [ ]:
df = pd.read_excel(path+"/Veri_Seti_10000.xlsx")
df = df.rename(columns={"Paylaşım": "Post", "Tespit": "Detected"})


Examines the basic statistical summary of the dataset (for numeric columns).


In [ ]:
df.describe()


Checks the structure of the dataset, column types, and missing value status.


In [ ]:
df.info()


Displays the records where the detection label is True along with all their columns.


In [ ]:
df.loc[df.Detected == True, list(df.columns)]


Shows the first five rows of the dataset as a preview.


In [ ]:
df.head(5)


Imports the required Scikit-learn components for classification, data splitting, evaluation, and TF-IDF vectorization.


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve,classification_report,hamming_loss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier


Separates the text column as features (X) and the label columns as targets (y), and splits the data into training/test sets.


In [ ]:
X = df["Post"]
y = df.drop("Post",axis=1,inplace=False)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


Defines a pipeline that converts texts into numeric features with TF-IDF and a multi-label classifier based on LinearSVC.


In [ ]:
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 3),
        min_df=2,
        max_features=25000,
        sublinear_tf=True
    )),
    ('clf', MultiOutputClassifier(
        LinearSVC(C=1.0, class_weight='balanced', random_state=42)
    ))
])


Trains the model on the training data and produces predictions on the test data.


In [ ]:
model_pipeline.fit(X_train.astype(str), y_train)

y_pred = model_pipeline.predict(X_test.astype(str))


Computes and prints per-class precision, recall, and F1 scores along with the Hamming Loss value.


In [ ]:
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=y.columns, zero_division=0))

print(f"Hamming Loss: {hamming_loss(y_test, y_pred):.4f}")


Imports visualization libraries and configures plot styling.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import multilabel_confusion_matrix

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


Visualizes each class's F1 score with a bar chart.


In [ ]:
classes = list(y.columns)
f1_scores = [0.88, 0.82, 0.85, 0.73, 1.00, 0.68, 0.97] # Values from the classification report

plt.figure(figsize=(10, 5))
sns.barplot(x=classes, y=f1_scores, palette="viridis")
plt.title("F1-Scores by Class", fontsize=14)
plt.xlabel("Classes", fontsize=12)
plt.ylabel("F1-Score", fontsize=12)
plt.ylim(0, 1.05)
for i, v in enumerate(f1_scores):
    plt.text(i, v + 0.01, f"{v:.2f}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


Shows the class distribution in the test set with a bar chart.


In [ ]:
class_counts = y_test.sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=class_counts.values, y=class_counts.index, palette="magma")
plt.title("Class Distribution in Test Data (Support)", fontsize=14)
plt.xlabel("Number of Samples", fontsize=12)
plt.ylabel("Classes", fontsize=12)
plt.tight_layout()
plt.show()


Draws multi-label confusion matrices as heatmaps for each label.


In [ ]:
mcm = multilabel_confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(4, 2, figsize=(15, 20))
axes = axes.ravel()

for i, label in enumerate(classes):
    sns.heatmap(mcm[i], annot=True, fmt='d', ax=axes[i], cmap='Blues', cbar=False,
                xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
    axes[i].set_title(f"{label} - Confusion Matrix", fontsize=12)
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')

if len(classes) < len(axes):
    fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()


Prints the categories predicted by the model for an example tweet text.


In [ ]:
ornek_metin = "what is this what is this WHAT IS THIS"

tahmin = model_pipeline.predict([ornek_metin])[0]

print(f"Input Text: '{ornek_metin}'\n")
print("Predicted Categories:")

eslesen_kategori_var_mi = False
for kategori, sonuc in zip(y.columns, tahmin):
    durum = "YES" if sonuc == 1 else "NO"
    if sonuc == 1:
        print(f"- {kategori}: {durum}")
        eslesen_kategori_var_mi = True

if not eslesen_kategori_var_mi:
    print("- Did not match any category (Normal content)")


Visualizes the LinearSVC coefficients of the TF-IDF features to show the most influential words for each class.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

tfidf = model_pipeline.named_steps['tfidf']
clf = model_pipeline.named_steps['clf']
feature_names = np.array(tfidf.get_feature_names_out())

fig, axes = plt.subplots(4, 2, figsize=(16, 24))
axes = axes.ravel()

for i, (label, estimator) in enumerate(zip(y.columns, clf.estimators_)):
    coefs = estimator.coef_[0]

    top_indices = np.argsort(coefs)[-10:]
    bottom_indices = np.argsort(coefs)[:10]
    selected_indices = np.concatenate([bottom_indices, top_indices])

    selected_coefs = coefs[selected_indices]
    selected_features = feature_names[selected_indices]

    colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in selected_coefs]

    sns.barplot(x=selected_coefs, y=selected_features, palette=colors, ax=axes[i], hue=selected_features, legend=False)
    axes[i].set_title(f"{label} Class - Most Influential Words/Features", fontsize=12, fontweight='bold')
    axes[i].set_xlabel("Weight (Coefficient Value)", fontsize=10)
    axes[i].axvline(0, color='black', linestyle='--', linewidth=1)

if len(y.columns) < len(axes):
    fig.delaxes(axes[-1])

plt.suptitle("Linear SVM Most Important Word Coefficient Distribution", fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
